# Description

This notebook generates a keywords clustering of the studies. The output file is in EPS format.

# Requirements

An environment with:
- matplotlib
- pandas
- openpyxl
- scikit-learn
- sentence-transformers

Use the following to install the packages:

```pip install -e ".[keyword_clustering]"```

# Prerequisites

- An excel file ```filename``` containing the references in ```sheet_name``` with at least the column ```column_name```.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import random
import numpy as np

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
filename = '../data/refs.xlsx'
sheet_name = 'RAW'
output_filename = "keywords_clustering.eps"
column_name = 'Keywords'
seed = 42
random.seed(seed)
np.random.seed(seed)

In [ ]:
data = pd.read_excel(filename, sheet_name=sheet_name)
data = data[~data[column_name].isna()]
df = data.rename(columns={column_name: 'keywords'})

In [ ]:
keywords = []
for kw_list in df["keywords"].dropna():
    for kw in kw_list.split(";"):
        kw = kw.strip().lower()
        if kw: 
            keywords.append(kw)

unique_keywords = sorted(set(keywords))


model = SentenceTransformer("all-MiniLM-L6-v2")
X = model.encode(unique_keywords, convert_to_numpy=True)  # déjà par défaut True




In [ ]:
from sklearn.mixture import GaussianMixture
import numpy as np

bic_scores = []
k_range = range(2, 10)
pca_bic = PCA(n_components=40, random_state=seed)
X_reduced = pca_bic.fit_transform(X)

for k in k_range:
    gmm = GaussianMixture(n_components=k, random_state=seed, n_init=5)
    gmm.fit(X_reduced)
    bic_scores.append(gmm.bic(X_reduced))

# Plot BIC curve
plt.figure(figsize=(7, 4))
plt.plot(k_range, bic_scores, marker='o')
plt.xlabel("Number of clusters k")
plt.ylabel("BIC")
plt.title("BIC by number of clusters")
plt.tight_layout()
plt.show()

best_k_bic = k_range[np.argmin(bic_scores)]
print(f"Best k according to BIC: {best_k_bic}")

In [ ]:
from sklearn.metrics import silhouette_score

sil_scores = []
for k in k_range:
    km = KMeans(n_clusters=k, random_state=seed, n_init=10)
    lbl = km.fit_predict(X)
    sil_scores.append(silhouette_score(X, lbl))

plt.figure(figsize=(7, 4))
plt.plot(k_range, sil_scores, marker='o', color='tab:orange')
plt.xlabel("Number of clusters k")
plt.ylabel("Silhouette score")
plt.title("Silhouette score by number of clusters")
plt.tight_layout()
plt.show()

best_k_sil = list(k_range)[np.argmax(sil_scores)]
print(f"Best k according to silhouette: {best_k_sil}")

In [ ]:
k = min(best_k_sil, best_k_bic)
kmeans = KMeans(n_clusters=k, random_state=seed, n_init=10)
labels = kmeans.fit_predict(X)

pca = PCA(n_components=2, random_state=seed)
coords = pca.fit_transform(X)

plot_df = pd.DataFrame({
    "keyword": unique_keywords,
    "x": coords[:, 0],
    "y": coords[:, 1],
    "cluster": labels
})

In [ ]:
from collections import Counter
from adjustText import adjust_text

keyword_counts = Counter(keywords)

fig, ax = plt.subplots(figsize=(16, 11))

clusters_names = sorted(plot_df["cluster"].unique())

for cluster_id in clusters_names:
    subset = plot_df[plot_df["cluster"] == cluster_id]
    ax.scatter(subset["x"], subset["y"], label=f"Cluster {cluster_id}", alpha=0.7, s=40)

    cx, cy = subset["x"].mean(), subset["y"].mean()
    #ax.text(cx, cy, f"Cluster {cluster_id}", fontsize=13, weight="bold", zorder=5,
    #        ha="center", va="center",
    #        #bbox=dict(facecolor="white", alpha=0.9, edgecolor="grey", boxstyle="round,pad=0.3"))

top_per_cluster = (
    plot_df.groupby("cluster", group_keys=False)
           .apply(lambda g: g.head(10))
)

texts = [
    ax.text(row["x"], row["y"], row["keyword"], fontsize=12)
    for _, row in top_per_cluster.iterrows()
]
adjust_text(
    texts,
    ax=ax,
    arrowprops=dict(arrowstyle="-", color="grey", alpha=0.4, lw=0.5)
)

ax.legend(loc='upper left', framealpha=1.0)#0.9)
ax.set_xlabel("PCA Component 1")
ax.set_ylabel("PCA Component 2")
plt.tight_layout(pad=3)#1.5)

plt.savefig(output_filename, bbox_inches='tight')
plt.show()

In [ ]:

clusters_keywords = {}
for cluster_id in sorted(plot_df["cluster"].unique()):
    clusters_keywords[cluster_id] = plot_df[plot_df["cluster"] == cluster_id]["keyword"].tolist()

for cid, kws in clusters_keywords.items():
    print(f"\n=== Cluster {cid} ===")
    for kw in kws:
        print(f" - {kw}")